# Session 1: 환경 세팅 & 데이터 탐색

## MLOps란?

**MLOps (Machine Learning Operations)** 는 머신러닝 모델을 개발부터 배포, 모니터링까지 체계적으로 관리하는 방법론입니다.

### 이 과정에서 만들 것

우리는 **대출 심사 자동화 시스템**을 처음부터 끝까지 구축합니다:

1. **데이터 탐색** (Session 1) - 데이터를 이해하고 인사이트 도출
2. **모델 학습** (Session 2) - XGBoost 모델 학습 및 Pipeline 구축
3. **API 기초** (Session 3) - FastAPI로 REST API 기본 구조 학습
4. **예측 API 구현** (Session 4) - 학습된 모델을 API로 서빙
5. **컨테이너화** (Session 5) - Docker로 패키징
6. **배포 & 모니터링** (Session 6) - 클라우드 배포 및 운영

### 왜 MLOps가 중요한가?

- 노트북에서 잘 동작하는 모델이 실제 서비스에서는 동작하지 않는 경우가 많습니다.
- 모델 재학습, 버전 관리, 성능 모니터링 등 운영 이슈를 체계적으로 다뤄야 합니다.
- 이 과정을 통해 **모델 개발 → 서빙 → 운영**의 전체 흐름을 경험합니다.

In [ ]:
!pip install fastapi uvicorn scikit-learn xgboost pandas joblib matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('환경 설정 완료!')

환경 설정 완료!


## 대출 심사 데이터 소개

### 비즈니스 시나리오

우리는 은행의 **대출 심사 자동화 시스템**을 구축하는 ML 엔지니어입니다.

현재 대출 심사는 심사역이 수동으로 처리하고 있어 시간이 오래 걸리고, 일관성이 부족합니다.  
ML 모델을 활용하여 대출 승인 여부를 예측하는 API를 만들어 심사 프로세스를 자동화하려 합니다.

### 데이터 컬럼 설명

| 컬럼 | 설명 | 타입 |
|------|------|------|
| 나이 | 신청자 나이 | 수치형 |
| 성별 | 남/여 | 범주형 |
| 연소득 | 연간 소득 (만원) | 수치형 |
| 근속연수 | 현 직장 근속 기간 (년) | 수치형 |
| 주거형태 | 자가/전세/월세 | 범주형 |
| 신용점수 | 신용 평가 점수 | 수치형 |
| 기존대출건수 | 기존 대출 수 | 수치형 |
| 연간카드사용액 | 연간 카드 사용 금액 (만원) | 수치형 |
| 부채비율 | 소득 대비 부채 비율 (%) | 수치형 |
| 대출신청액 | 신청한 대출 금액 (만원) | 수치형 |
| 대출목적 | 대출 사유 | 범주형 |
| 상환방식 | 원금균등/원리금균등 등 | 범주형 |
| 대출기간 | 대출 기간 (개월) | 수치형 |
| 승인여부 | 대출 승인=1, 거절=0 (타겟) | 이진 |

In [17]:
# 데이터 불러오기
df = pd.read_csv("../data/loan_data.csv")

# print(df)
# print(df.shape) # (1500, 14)
# print(df.head(2))
print(df.describe())
# print(df['승인여부'].value_counts())


                나이           연소득         근속연수         신용점수       기존대출건수  \
count  1500.000000   1500.000000  1500.000000  1500.000000  1500.000000   
mean     38.078667   3884.866667     6.074667   568.626667     1.258000   
std       8.582633   1521.267175     5.136927    86.603931     1.136206   
min      23.000000   1500.000000     0.000000   300.000000     0.000000   
25%      32.000000   2800.000000     2.000000   510.750000     0.000000   
50%      38.000000   3650.000000     5.000000   569.000000     1.000000   
75%      44.000000   4700.000000     9.000000   626.000000     2.000000   
max      65.000000  13900.000000    30.000000   900.000000     5.000000   

           연간카드사용액         부채비율         대출신청액         대출기간         승인여부  
count  1500.000000  1500.000000   1500.000000  1500.000000  1500.000000  
mean   1757.760000    37.353867   3418.533333    36.840000     0.608667  
std     995.241538    15.070179   1825.101964    13.990886     0.488211  
min     280.000000     3.000

In [20]:
# 결측치 처리
df.isnull().sum()

나이         0
성별         0
연소득        0
근속연수       0
주거형태       0
신용점수       0
기존대출건수     0
연간카드사용액    0
부채비율       0
대출신청액      0
대출목적       0
상환방식       0
대출기간       0
승인여부       0
dtype: int64

In [22]:
# 클래스 불균형 분포 확인
df['승인여부'].value_counts()


승인여부
1    913
0    587
Name: count, dtype: int64

In [23]:
# 문자 데이터 -> 숫자 데이터로 변환
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   나이       1500 non-null   int64  
 1   성별       1500 non-null   object 
 2   연소득      1500 non-null   int64  
 3   근속연수     1500 non-null   int64  
 4   주거형태     1500 non-null   object 
 5   신용점수     1500 non-null   int64  
 6   기존대출건수   1500 non-null   int64  
 7   연간카드사용액  1500 non-null   int64  
 8   부채비율     1500 non-null   float64
 9   대출신청액    1500 non-null   int64  
 10  대출목적     1500 non-null   object 
 11  상환방식     1500 non-null   object 
 12  대출기간     1500 non-null   int64  
 13  승인여부     1500 non-null   int64  
dtypes: float64(1), int64(9), object(4)
memory usage: 164.2+ KB


In [26]:
df['성별'].unique()

array(['여', '남'], dtype=object)

In [27]:
df['주거형태'].unique()

array(['전세', '월세', '자가'], dtype=object)

In [29]:
df['대출목적'].unique()

array(['자동차', '주택구입', '전세자금', '생활비', '사업자금', '의료비', '교육비'], dtype=object)

In [ ]:
df['사']

In [18]:
# AI를 만들기 위해서 데이터로 모델을 학습을 시켜야 한다.




0       1
1       0
2       1
3       0
4       0
       ..
1495    1
1496    1
1497    0
1498    1
1499    1
Name: 승인여부, Length: 1500, dtype: int64


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


## 데이터 탐색 인사이트 정리

### 발견한 주요 사항

1. **타겟 분포**: 승인여부의 분포를 확인하여 클래스 불균형 여부를 파악했습니다.
2. **신용점수**: 승인 그룹이 거절 그룹보다 신용점수가 높은 경향이 뚜렷합니다. → 중요한 피처
3. **연소득**: 승인 그룹의 연소득이 상대적으로 높습니다.
4. **부채비율**: 부채비율이 높을수록 거절될 가능성이 높습니다.
5. **범주형 변수**: 주거형태, 대출목적 등에 따라 승인율 차이가 존재합니다.

